In [15]:
import pandas as pd
import numpy as np
import os.path as op
from scipy import stats

## Load Data

In [16]:
# Load the table files to get subject IDs
drawn_table_fn = "./dset/group-drawn/habenula/sub-group_task-rest_desc-1S2StTesthabenula_table.txt"
avg_table_fn = "./dset/group-avg/habenula/sub-group_task-rest_desc-1S2StTesthabenula_table.txt"

# Load participants.tsv
participants_fn = "./dset/participants.tsv"
participants_df = pd.read_csv(participants_fn, sep="\t", low_memory=False)

print(f"Total participants in dataset: {len(participants_df)}")

Total participants in dataset: 2156


## Extract Subjects from Drawn Habenula Analysis

In [17]:
# Load drawn table
drawn_table_df = pd.read_csv(drawn_table_fn, sep="\t")
drawn_subjects = drawn_table_df["Subj"].tolist()

print(f"Subjects in drawn habenula analysis: {len(drawn_subjects)}")
print(f"First 10 subjects: {drawn_subjects[:10]}")

Subjects in drawn habenula analysis: 1479
First 10 subjects: ['sub-29013', 'sub-29099', 'sub-29477', 'sub-29731', 'sub-0050695', 'sub-29319', 'sub-29118', 'sub-0050032', 'sub-0051110', 'sub-29468']


## Extract Demographics for Drawn Subjects

In [18]:
# Filter participants to only those in drawn analysis
drawn_demo_df = participants_df[
    participants_df["participant_id"].isin(drawn_subjects)
].copy()

# Select only the columns of interest
columns_of_interest = [
    "participant_id",
    "DX_GROUP",
    "AGE_AT_SCAN",
    "SEX",
    "HANDEDNESS_CATEGORY",
    "DSM_IV_TR",
    "COMORBIDITY",
    "CURRENT_MED_STATUS",
]

drawn_demo_df = drawn_demo_df[columns_of_interest]

# Convert DX_GROUP to numeric and replace with labels
drawn_demo_df["DX_GROUP"] = pd.to_numeric(drawn_demo_df["DX_GROUP"], errors="coerce")
drawn_demo_df["DX_GROUP_label"] = drawn_demo_df["DX_GROUP"].map(
    {1: "ASD", 2: "TD"}
)

# Convert age to numeric
drawn_demo_df["AGE_AT_SCAN"] = pd.to_numeric(
    drawn_demo_df["AGE_AT_SCAN"], errors="coerce"
)

print(f"\nDemographics extracted for {len(drawn_demo_df)} subjects")
drawn_demo_df.head(10)


Demographics extracted for 1479 subjects


,participant_id,DX_GROUP,AGE_AT_SCAN,SEX,HANDEDNESS_CATEGORY,DSM_IV_TR,COMORBIDITY,CURRENT_MED_STATUS,DX_GROUP_label
2,sub-0050004,1,19.09,1,R,1.0,NaN,0,ASD
3,sub-0050005,1,13.73,2,R,1.0,NaN,1,ASD
4,sub-0050006,1,13.37,1,L,1.0,NaN,0,ASD
5,sub-0050007,1,17.78,1,R,1.0,NaN,0,ASD
6,sub-0050008,1,32.45,1,R,1.0,NaN,1,ASD
8,sub-0050010,1,35.20,1,L,1.0,NaN,0,ASD
9,sub-0050011,1,16.93,1,L,1.0,NaN,0,ASD
10,sub-0050012,1,21.48,1,R,1.0,NaN,0,ASD
11,sub-0050013,1,9.33,1,R,1.0,NaN,0,ASD
12,sub-0050014,1,14.20,1,R,1.0,NaN,1,ASD


## Calculate Summary Statistics by Diagnosis Group

In [19]:
def summarize_demographics(df, group_col="DX_GROUP_label"):
    """Calculate summary statistics for each diagnosis group."""
    summary_lines = []
    summary_lines.append("=" * 80)
    summary_lines.append("DEMOGRAPHIC SUMMARY - DRAWN HABENULA SUBJECTS")
    summary_lines.append("=" * 80)
    summary_lines.append("")
    
    # Overall sample size
    summary_lines.append(f"Total Sample Size: {len(df)}")
    summary_lines.append("")
    
    # Group by diagnosis
    for group in ["ASD", "TD"]:
        group_df = df[df[group_col] == group].copy()
        n = len(group_df)
        
        summary_lines.append("-" * 80)
        summary_lines.append(f"{group} GROUP (n = {n})")
        summary_lines.append("-" * 80)
        summary_lines.append("")
        
        # Age: mean (SD)
        age_mean = group_df["AGE_AT_SCAN"].mean()
        age_std = group_df["AGE_AT_SCAN"].std()
        summary_lines.append(
            f"Age: {age_mean:.2f} ({age_std:.2f}) years [mean (SD)]"
        )
        summary_lines.append("")
        
        # Sex: counts only
        summary_lines.append("Sex:")
        sex_counts = group_df["SEX"].value_counts()
        for sex_val, count in sex_counts.items():
            sex_label = "Male" if sex_val == 1 else "Female" if sex_val == 2 else f"Unknown ({sex_val})"
            summary_lines.append(f"  {sex_label}: {count}")
        summary_lines.append("")
        
        # Handedness: counts only
        summary_lines.append("Handedness:")
        hand_counts = group_df["HANDEDNESS_CATEGORY"].value_counts(dropna=False)
        for hand_val, count in hand_counts.items():
            hand_label = str(hand_val) if pd.notna(hand_val) else "Missing"
            summary_lines.append(f"  {hand_label}: {count}")
        summary_lines.append("")
        
        # DSM_IV_TR: counts only
        summary_lines.append("DSM-IV-TR Diagnosis:")
        dsm_counts = group_df["DSM_IV_TR"].value_counts(dropna=False)
        for dsm_val, count in dsm_counts.items():
            dsm_label = str(dsm_val) if pd.notna(dsm_val) else "Missing"
            summary_lines.append(f"  {dsm_label}: {count}")
        summary_lines.append("")
        
        # Comorbidity: counts only
        summary_lines.append("Comorbidity:")
        comor_counts = group_df["COMORBIDITY"].value_counts(dropna=False)
        for comor_val, count in comor_counts.items():
            comor_label = str(comor_val) if pd.notna(comor_val) else "Missing"
            summary_lines.append(f"  {comor_label}: {count}")
        summary_lines.append("")
        
        # Medication status: counts only
        summary_lines.append("Current Medication Status:")
        med_counts = group_df["CURRENT_MED_STATUS"].value_counts(dropna=False)
        for med_val, count in med_counts.items():
            if pd.notna(med_val):
                med_label = (
                    "No medication" if med_val == 0 
                    else "Medication" if med_val == 1 
                    else f"Other ({med_val})"
                )
            else:
                med_label = "Missing"
            summary_lines.append(f"  {med_label}: {count}")
        summary_lines.append("")
        summary_lines.append("")
    
    summary_lines.append("=" * 80)
    
    return "\n".join(summary_lines)


# Generate summary
summary_text = summarize_demographics(drawn_demo_df)
print(summary_text)

DEMOGRAPHIC SUMMARY - DRAWN HABENULA SUBJECTS

Total Sample Size: 1479

--------------------------------------------------------------------------------
ASD GROUP (n = 661)
--------------------------------------------------------------------------------

Age: 16.68 (8.23) years [mean (SD)]

Sex:
  Male: 572
  Female: 89

Handedness:
  1.0: 255
  R: 195
  Missing: 131
  L: 29
  2.0: 26
  3.0: 20
  -9999: 2
  Ambi: 2
  Mixed: 1

DSM-IV-TR Diagnosis:
  Missing: 313
  1.0: 222
  2.0: 66
  3.0: 28
  -9999.0: 20
  0.0: 12

Comorbidity:
  Missing: 624
  ADHD Inattentive: 5
  Dysthymia: 3
  Mood Disorder NOS: 3
  ADHD NOS: 2
  Generalized Anxiety Disorder: 2
  Mood Disorder NOS; Separation Anxiety Dx; Enuresis: 1
  Social Phobia;: 1
  ADHD Combined: 1
  Dysthymia; Agoraphobia dx;: 1
  Specific Phobia: needles/shots: 1
  Disruptive disorder NOS: 1
  Specific Phobia: Butterflies: 1
  Generalized Anxiety Disorder; Specific phobia; Enuresis; Encopresis: 1
  Anxiety Disorder NOS & Depressive Disord

In [20]:
def calculate_group_comparisons(df):
    """Calculate statistical tests comparing ASD vs TD groups."""
    
    asd_df = df[df["DX_GROUP_label"] == "ASD"].copy()
    td_df = df[df["DX_GROUP_label"] == "TD"].copy()
    
    results = []
    
    # Age: Independent samples t-test
    asd_age = asd_df["AGE_AT_SCAN"].dropna()
    td_age = td_df["AGE_AT_SCAN"].dropna()
    
    if len(asd_age) > 0 and len(td_age) > 0:
        t_stat, p_val = stats.ttest_ind(asd_age, td_age)
        results.append({
            "Variable": "Age",
            "Test": "Independent t-test",
            "Statistic": f"t = {t_stat:.3f}",
            "p-value": f"{p_val:.4f}",
            "Significant": "Yes" if p_val < 0.05 else "No"
        })
    
    # Sex: Chi-square test
    sex_crosstab = pd.crosstab(df["DX_GROUP_label"], df["SEX"])
    if sex_crosstab.shape[0] > 1 and sex_crosstab.shape[1] > 1:
        chi2, p_val, dof, expected = stats.chi2_contingency(sex_crosstab)
        results.append({
            "Variable": "Sex",
            "Test": "Chi-square",
            "Statistic": f"χ² = {chi2:.3f} (df={dof})",
            "p-value": f"{p_val:.4f}",
            "Significant": "Yes" if p_val < 0.05 else "No"
        })
    
    # Handedness: Chi-square test (excluding missing values)
    hand_df = df[df["HANDEDNESS_CATEGORY"].notna()].copy()
    if len(hand_df) > 0:
        hand_crosstab = pd.crosstab(hand_df["DX_GROUP_label"], hand_df["HANDEDNESS_CATEGORY"])
        if hand_crosstab.shape[0] > 1 and hand_crosstab.shape[1] > 1:
            chi2, p_val, dof, expected = stats.chi2_contingency(hand_crosstab)
            results.append({
                "Variable": "Handedness",
                "Test": "Chi-square",
                "Statistic": f"χ² = {chi2:.3f} (df={dof})",
                "p-value": f"{p_val:.4f}",
                "Significant": "Yes" if p_val < 0.05 else "No"
            })
    
    # DSM_IV_TR: Chi-square test (excluding missing values)
    dsm_df = df[df["DSM_IV_TR"].notna()].copy()
    if len(dsm_df) > 0:
        dsm_crosstab = pd.crosstab(dsm_df["DX_GROUP_label"], dsm_df["DSM_IV_TR"])
        if dsm_crosstab.shape[0] > 1 and dsm_crosstab.shape[1] > 1:
            chi2, p_val, dof, expected = stats.chi2_contingency(dsm_crosstab)
            results.append({
                "Variable": "DSM-IV-TR",
                "Test": "Chi-square",
                "Statistic": f"χ² = {chi2:.3f} (df={dof})",
                "p-value": f"{p_val:.4f}",
                "Significant": "Yes" if p_val < 0.05 else "No"
            })
    
    # Comorbidity: Chi-square test (excluding missing values)
    comor_df = df[df["COMORBIDITY"].notna()].copy()
    if len(comor_df) > 0:
        comor_crosstab = pd.crosstab(comor_df["DX_GROUP_label"], comor_df["COMORBIDITY"])
        if comor_crosstab.shape[0] > 1 and comor_crosstab.shape[1] > 1:
            chi2, p_val, dof, expected = stats.chi2_contingency(comor_crosstab)
            results.append({
                "Variable": "Comorbidity",
                "Test": "Chi-square",
                "Statistic": f"χ² = {chi2:.3f} (df={dof})",
                "p-value": f"{p_val:.4f}",
                "Significant": "Yes" if p_val < 0.05 else "No"
            })
    
    # Medication status: Chi-square test (excluding missing values)
    med_df = df[df["CURRENT_MED_STATUS"].notna()].copy()
    if len(med_df) > 0:
        med_crosstab = pd.crosstab(med_df["DX_GROUP_label"], med_df["CURRENT_MED_STATUS"])
        if med_crosstab.shape[0] > 1 and med_crosstab.shape[1] > 1:
            chi2, p_val, dof, expected = stats.chi2_contingency(med_crosstab)
            results.append({
                "Variable": "Medication Status",
                "Test": "Chi-square",
                "Statistic": f"χ² = {chi2:.3f} (df={dof})",
                "p-value": f"{p_val:.4f}",
                "Significant": "Yes" if p_val < 0.05 else "No"
            })
    
    return pd.DataFrame(results)


# Calculate statistical comparisons
stats_df = calculate_group_comparisons(drawn_demo_df)
print("\n" + "=" * 80)
print("STATISTICAL COMPARISONS: ASD vs TD")
print("=" * 80)
print(stats_df.to_string(index=False))
print("\nNote: p < 0.05 indicates statistically significant group difference")


STATISTICAL COMPARISONS: ASD vs TD
         Variable               Test           Statistic p-value Significant
              Age Independent t-test           t = 0.815  0.4150          No
              Sex         Chi-square  χ² = 31.286 (df=1)  0.0000         Yes
       Handedness         Chi-square  χ² = 11.055 (df=7)  0.1362          No
        DSM-IV-TR         Chi-square χ² = 686.078 (df=4)  0.0000         Yes
Medication Status         Chi-square χ² = 198.297 (df=5)  0.0000         Yes

Note: p < 0.05 indicates statistically significant group difference


## Statistical Comparisons Between Groups

## Save Results

In [21]:
# Save demographics dataframe
output_csv = "./dset/group-drawn/habenula/demographics_drawn.csv"
drawn_demo_df.to_csv(output_csv, index=False)
print(f"Demographics dataframe saved to: {output_csv}")

# Save summary statistics
output_txt = "./dset/group-drawn/habenula/demographics_drawn_summary.txt"
with open(output_txt, "w") as f:
    f.write(summary_text)
    f.write("\n\n")
    f.write("=" * 80 + "\n")
    f.write("STATISTICAL COMPARISONS: ASD vs TD\n")
    f.write("=" * 80 + "\n")
    f.write(stats_df.to_string(index=False))
    f.write("\n\nNote: p < 0.05 indicates statistically significant group difference\n")
print(f"Summary statistics saved to: {output_txt}")

# Save statistical comparisons as CSV
stats_csv = "./dset/group-drawn/habenula/demographics_drawn_statistics.csv"
stats_df.to_csv(stats_csv, index=False)
print(f"Statistical comparisons saved to: {stats_csv}")

Demographics dataframe saved to: ./dset/group-drawn/habenula/demographics_drawn.csv
Summary statistics saved to: ./dset/group-drawn/habenula/demographics_drawn_summary.txt
Statistical comparisons saved to: ./dset/group-drawn/habenula/demographics_drawn_statistics.csv


## Optional: Compare with Average Habenula Subjects

In [22]:
# Load average table
avg_table_df = pd.read_csv(avg_table_fn, sep="\t")
avg_subjects = avg_table_df["Subj"].tolist()

print(f"Subjects in average habenula analysis: {len(avg_subjects)}")

# Check overlap
drawn_set = set(drawn_subjects)
avg_set = set(avg_subjects)

overlap = drawn_set.intersection(avg_set)
only_drawn = drawn_set - avg_set
only_avg = avg_set - drawn_set

print(f"\nOverlap between drawn and average: {len(overlap)} subjects")
print(f"Only in drawn: {len(only_drawn)} subjects")
print(f"Only in average: {len(only_avg)} subjects")

Subjects in average habenula analysis: 1584

Overlap between drawn and average: 1479 subjects
Only in drawn: 0 subjects
Only in average: 105 subjects
